In [1194]:
import pandas as pd
import numpy as np
import re
from urllib.parse import unquote


import sys
sys.path.append("./submission")
import clean

df = pd.read_csv("./data/test_features.csv")
df = df[['record_id','origin_station',
 'destination_station',
 'district',
 'encoded_transport',
 'day_of_week',
 'is_holiday',
 'weather_condition',
 'country_code']]

Lucciano: Cleaning Origin and Destination

In [1195]:
# Canonical station -> ordered regexes matched against the SQUASHED key
# (lowercased, url-decoded, wrappers stripped, all non-alphanumerics removed).
# Order matters: more specific patterns first. ^x$ pins the short codes.
_STATION_RULES = [
    ("Causeway Bay", r"causewaybay|cwbhub|^cwb$|^cb$"),
    ("Central", r"central|stncentral|censt|^cstation$|^cstn$|^cs$"),
    ("Admiralty", r"admiralt|admxfer|^adm$"),
    ("Sha Tin", r"shatin|shatinterm|^stin$|^stn$|^st$"),
    ("Mong Kok", r"mongkok|mkhub|^mk$"),
    ("Tsim Sha Tsui", r"tsimshatsui|tsteast|^tst$"),
    ("North Point", r"northpoint|northpt|npferry|^np$"),
    ("Tsuen Wan", r"tsuenwan|tswline|^twn$|^tswan$|^tw$"),
    ("Kennedy Town", r"kennedyt|ktownstop|^ktown$|^kt$"),
    ("Wan Chai", r"wanchai|wanchaistop|^wchai$|^wc$"),
]
# Markers/non-stations to skip. unk = unknown placeholder; depot/workshop/audithub
# are internal sites, not passenger stations -> treated as missing.
_NOISE = re.compile(r"^(unk|audithub|depot|workshop)$")


def clean_station(s: pd.Series) -> pd.Series:
    def clean_value(v):
        if not isinstance(v, str):
            return pd.NA
        x = unquote(unquote(v)).strip().lower()  # decode (handles %2520), trim
        for tok in re.split(r"[:|=\-]+", x):
            key = re.sub(r"[^a-z0-9]", "", tok)  # squash: drop spaces/./%20 etc
            if not key or _NOISE.match(key):
                continue
            for canon, pat in _STATION_RULES:
                if re.search(pat, key):
                    return canon
        return pd.NA  # no station token -> missing

    return s.map(clean_value)

df['origin_station'] = clean_station(df['origin_station'])
df['destination_station'] = clean_station(df['destination_station'])

Cleaning District

In [1196]:
# Clean district column values of prefix/suffix patterns

def clean_district(df):
    # Strip whitespace
    df['district'] = df['district'].str.strip()
    
    # Remove prefix patterns (e.g., "D123" → "123")
    prefix_pattern = r'(D|zone|hold|district|Legacy|region|Leg|LEG)[^a-zA-Z0-9]+'
    df['district'] = df['district'].str.replace(prefix_pattern, '', regex=True)
    
    # Remove suffix patterns (e.g., "central_zone" → "central")
    suffix_pattern = r'[_|](?:new|core|zone|cluster|sector|OPS)'
    df['district'] = df['district'].str.replace(suffix_pattern, '', regex=True)
    
    # Normalize missing/weird values into NaN
    unknown_values = [
        'Legacy', 'unknown', '??', 'mismatch', 'Null', 'legacy',
        'Internal', 'operations', 'audit', 'eh', 'unk', ''
    ]
    df['district'] = df['district'].str.replace(r'^(operations|audit|unk|eh)\s*$', '', regex=True, case=False)

    df['district'] = df['district'].replace(unknown_values, np.nan)

    return df

df = clean_district(df)

In [1197]:
#map district variants into unified category

district_variant_mapping = [
    ("Central and Western", r"cw|cwbhub|^cwb$|^cb$|central_and_western|central and western"),
    ("Tsuen Wan", r"tw|tsw|tsuen_wan|tsuen wan"),
    ("Yau Tsim Mong", r"YTM|ytm|yau tsim mong|yau_tsim_mong"),
    ("Sha Tin", r"ST|sha tin|shatin|st|sha_tin"),
    ("Eastern", r"eastern|east_harbour"), 
    ("Wan Chai", r"wanchai|wanchaistop|^wchai$|^wc$|wan chai|wan_chai")
]

def map_district(value):
    if pd.isna(value):
        return value
    value = str(value).lower().strip()
    for full_name, pattern in district_variant_mapping:
        if pattern and re.search(pattern, value):
            return full_name
    return value  # return original if no match

df['district'] = df['district'].apply(map_district)



In [1198]:
# map district NANs from known origin station:

station_district_mapping = {
    'Central and Western': ('Central', 'Admiralty', 'Kennedy Town'),
    'Wan Chai': ('Wan Chai', 'Causeway Bay'),
    'Yau Tsim Mong': ('Mong Kok', 'Tsim Sha Tsui'),
    'Sha Tin': ('Sha Tin',),
    'Tsuen Wan': ('Tsuen Wan',),
    'Eastern': ('North Point',)
}


def fill_district_from_station(df, station_col='origin_station', district_col='district'):
    # Build reverse lookup: station -> district
    station_to_district = {
        station.lower(): district
        for district, stations in station_district_mapping.items()
        for station in stations
    }
    
    # Fill nulls with station lookup
    df[district_col] = df[district_col].fillna(
        df[station_col].str.lower().str.strip().map(station_to_district)
    )
    return df

# Usage
df = fill_district_from_station(df)

df['district'].value_counts(dropna=False)

district
Central and Western    543
Yau Tsim Mong          431
Wan Chai               405
Sha Tin                281
Tsuen Wan              193
NaN                    127
Eastern                113
Name: count, dtype: int64

In [1199]:
df[['origin_station','destination_station','district']].sample(30)

,origin_station,destination_station,district
516,NaN,NaN,NaN
40,Kennedy Town,Mong Kok,Central and Western
1176,Central,Kennedy Town,Central and Western
429,Mong Kok,Central,Yau Tsim Mong
736,Tsim Sha Tsui,North Point,Yau Tsim Mong
150,Kennedy Town,Mong Kok,Central and Western
368,Tsim Sha Tsui,Causeway Bay,Yau Tsim Mong
1897,NaN,NaN,NaN
674,North Point,Kennedy Town,Eastern
1322,Kennedy Town,Admiralty,Central and Western


Cleaning Day of Week

In [1200]:
df['day_of_week'] = df['day_of_week'].str.title()
df['day_of_week'].value_counts(dropna=False)

day_of_week
Fri    366
Thu    329
Wed    324
Mon    308
Tue    290
Sat    281
Sun    195
Name: count, dtype: int64

In [1201]:
clean.SUBMISSION_FEATURES

['origin_station',
 'destination_station',
 'district',
 'transport_type',
 'transport_detail',
 'mode',
 'service_level',
 'operator',
 'day_of_week',
 'is_holiday',
 'weather_condition',
 'country_code']

Cleaning encode_transport

In [1202]:
# Parse encoded_transport into 5 columns
_SPECIAL_NO_TRANSPORT = {
    'maintenance_shift', 'admin_move', 'staff_shuttle', 'test_run',
    'unk', 'encoded_transport'
}

def parse_encoded_transport(s):
    if not isinstance(s, str):
        return {'transport_type': pd.NA, 'transport_detail': pd.NA,
                'mode': pd.NA, 'service_level': pd.NA, 'operator': pd.NA}
    v = s.strip().lower()
    if v in _SPECIAL_NO_TRANSPORT or not v:
        return {'transport_type': pd.NA, 'transport_detail': pd.NA,
                'mode': pd.NA, 'service_level': pd.NA, 'operator': pd.NA}

    # transport_type - use flexible separators, handle word boundaries around them
    m = re.search(r'(?:^|[_\-|\.::=])(tram|ferry|bus)(?:[_\-|\.::]|$)', v)
    if not m:
        # Try concatenated at start (e.g., buslocstdkowloon)
        m = re.search(r'^(tram|ferry|bus)(loc|exp)(std|prem)', v)
    if m:
        transport_type = m.group(1)
    else:
        transport_type = pd.NA

    # transport_detail
    m = re.search(r'(?:^|[_\-|\.::=])(tram|ferry|bus)[_\-]?(apt|ngt|night|xhbr)', v)
    if m:
        detail_map = {'apt': 'airport', 'ngt': 'night', 'night': 'night', 'xhbr': 'crossharbour'}
        transport_detail = detail_map.get(m.group(2), 'general')
    else:
        transport_detail = 'general'

    # mode - check multiple patterns
    m = re.search(r'\b(?:run|mode)[=:](\w+)', v)
    if m:
        mode_map = {'loc': 'local', 'exp': 'express'}
        mode = mode_map.get(m.group(1), pd.NA)
    else:
        # Try standalone patterns with various separators: _loc_, .loc., |loc|, ::loc::
        m = re.search(r'(?:^|[_\-|\.::])(loc|exp)(?:[_\-|\.::]|$)', v)
        if m:
            mode = 'local' if m.group(1) == 'loc' else 'express'
        else:
            # Concatenated: buslocstdkowloon, busexppremcitybus
            m = re.search(r'(tram|ferry|bus)(loc|exp)(std|prem)', v)
            if m:
                mode = 'local' if m.group(2) == 'loc' else 'express'
            else:
                mode = pd.NA

    # service_level - check multiple patterns
    m = re.search(r'\b(?:tier|svc)[=:](\w+)', v)
    if m:
        sl_map = {'std': 'standard', 'prem': 'premium', 'missing': pd.NA}
        service_level = sl_map.get(m.group(1), pd.NA)
    else:
        # Try standalone patterns with various separators: _std_, .std., |std|, ::std::
        m = re.search(r'(?:^|[_\-|\.::,])(std|prem)(?:[_\-|\.::]|$)', v)
        if m:
            service_level = 'standard' if m.group(1) == 'std' else 'premium'
        else:
            # Concatenated: buslocstdkowloon, busexppremcitybus
            m = re.search(r'(tram|ferry|bus)(loc|exp)(std|prem)', v)
            if m:
                service_level = 'standard' if m.group(3) == 'std' else 'premium'
            else:
                service_level = pd.NA

    # operator
    m = re.search(r'\b(?:op|owner)[=:](\w+)', v)
    if m:
        operator = m.group(1)
    else:
        m = re.search(r'(?<![|:])(\w+)[-|_]?(tail|delta|v2|\?|\?\?|c)$', v)
        if m:
            operator = m.group(1)
        else:
            parts = re.split(r'[:_|\[\];=,\.\-]+', v)
            parts = [p for p in parts if p not in ('', '??','unk', 'unknown', 'l2)', 'tail','ops', 'srcfeed', 'l2', 'amb', 'legacy', 'feed', 'type', 'mode', 'run', 'tier', 'svc', 'op', 'owner', 'sys', 'batch', 'src', 'kind')]
            operator = parts[-1] if parts else pd.NA
            if operator in ('std', 'prem', 'loc', 'exp', 'unk', ''):
                operator = pd.NA

    # Handle concatenated patterns like buslocstdkowloon -> extract operator at end
    m = re.search(r'(citybus|kowloon|ferryhk)$', str(operator))
    if m:
        operator = m.group(1)

    # Fix truncated operators (missing first letter)
    if operator == 'itybus':
        operator = 'citybus'
    elif operator == 'owloon':
        operator = 'kowloon'
    elif operator == 'erryhk':
        operator = 'ferryhk'

    # Normalize garbage/invalid operators to NaN
    invalid_operators = {'unknown', 'delta', 'endcap)(batch', ''}
    if operator in invalid_operators:
        operator = pd.NA

    return {'transport_type': transport_type, 'transport_detail': transport_detail,
            'mode': mode, 'service_level': service_level, 'operator': operator}

parsed = df['encoded_transport'].apply(parse_encoded_transport).apply(pd.Series)
df = pd.concat([df, parsed], axis=1)

df[['encoded_transport', 'transport_type',
 'transport_detail',
 'mode',
 'service_level',
 'operator']].sample(20)

,encoded_transport,transport_type,transport_detail,mode,service_level,operator
229,UNK::bus::loc::kowloon,bus,general,local,NaN,kowloon
1861,"kind,tram-apt,mode,loc,tier,std,op,citybus",NaN,general,NaN,NaN,citybus
1715,srcfeed::bus::loc::prem::citybus::delta,bus,general,local,premium,NaN
226,bus-apt__MISS__citybus,bus,airport,NaN,NaN,citybus
1685,type=bus-xhbr;mode=exp;tier=missing;op=unknown,bus,crossharbour,express,NaN,NaN
930,feed=OPS|kind=bus|run=loc|tier=prem|op=citybus,bus,general,local,premium,citybus
1397,bus-loc-std-citybus-c,bus,general,local,standard,citybus
1118,ferry-apt.loc.std.ferryhk,ferry,airport,local,standard,ferryhk
1592,"kind,tram,mode,loc,tier,std,op,citybus",NaN,general,NaN,NaN,citybus
709,UNK::bus::loc::citybus,bus,general,local,NaN,citybus


In [1203]:
df['mode'].value_counts(dropna=False).head(30)

mode
local      1097
NaN         651
express     345
Name: count, dtype: int64

In [1204]:
df.drop(columns='encoded_transport', inplace=True)
df

,record_id,origin_station,destination_station,district,day_of_week,is_holiday,weather_condition,country_code,transport_type,transport_detail,mode,service_level,operator
0,R08001,Sha Tin,Tsuen Wan,Sha Tin,Fri,0,NaN,UNK,tram,general,local,standard,NaN
1,R08002,Tsuen Wan,Kennedy Town,Tsuen Wan,Thu,0,NaN,852,tram,crossharbour,local,NaN,NaN
2,R08003,Causeway Bay,Wan Chai,Wan Chai,Wed,0,NaN,territory-hk,NaN,general,local,standard,citybus
3,R08004,Admiralty,Causeway Bay,Central and Western,Thu,0,wx_unknown,legacy,NaN,general,express,standard,citybus
4,R08005,North Point,Central,Sha Tin,Sat,1,NaN,legacy,bus,general,express,standard,citybus
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2088,R09170,NaN,NaN,Sha Tin,Sun,0,SUNNY,TST,NaN,NaN,NaN,NaN,NaN
2089,R08975,Wan Chai,Central,Yau Tsim Mong,Sat,0,SUNNY,cc=852|OPS,tram,night,express,standard,citybus
2090,R09878,Sha Tin,Tsuen Wan,Wan Chai,Tue,0,heavy-rn,legacy-null,ferry,general,NaN,NaN,ferryhk
2091,R08338,Mong Kok,Wan Chai,Yau Tsim Mong,Mon,0,heavy-rn,territory-hk,tram,general,express,standard,citybus
